**VAE : Variational Auto Encoder**
A generative model that consistes of an Encoder ( which compress the input)
and a decoder (which reconstructs or decompress the input from the compressed data)

In [ ]:
import torch # import pytorch library
import torch.nn as nn # for building layers in the neural nw
import torch.optim as optim # for optimization algos
import torchvision # pytorch computer vision library for img data
import torchvision.transforms as transforms # image transformations
from torch.utils.data import DataLoader # load data in batches
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from torch.nn.modules.activation import ReLU
#define a class for our VAE extending the neural network module class of pytorch
class VAE(nn.Module):
  def __init__(self, input_dim = 784, hidden_dim= 400, latent_dim = 20):
    # initialize the VAE parameters
    # input_dim  : no. of input features (28x28 = 784 for mnist)
    # hidden_dim : no. of nodes in hidden layer (half of ip)
    # latent_dim : no. of dimension in latent space (compressed)

    #call the constructor of the parent class
    super(VAE, self).__init__()


    #ENCODER
    ############
    #create an encoder converts ip into latent dims
    #create an encoder layer object of Sequential class
    self.encoder = nn.Sequential(
        nn.Linear(input_dim, hidden_dim)
        #first fully connected layer with Linear activation
        nn.ReLU() # relu activation
    )

    #create a seperate layer for learning mean
    self.mu_layer = nn.Linear(hidden_dim, latent_dim)
    #create another seperate layer for learning variance
    self.log_var_layer = nn.Linear(hidden_dim, latent_dim)


    #DECODER NETWORK
    ################
    #create an decoder converts latent dims into output
    self.decoder = nn.Sequential(
        nn.Linear(input_dim, hidden_dim)
        #first fully connected layer with Linear activation
        nn.ReLU(), # relu activation
        nn.Linear(hidden_dim, input_dim),
        #to make sure the output values are between zero and one
        #use a sigmoid activation fn
        nn.Sigmoid()
    )

  #within the class define the fn for reparametrization
  def reparameterize(self,my, log_var):
    #randomness z = mean +std * random noise epsilon
    std = torch.exp(0.5*log_var)
    epsilotn = torch.randn_like(std)
    return my + std*epsilotn

  #fn for forward pass of the neural network(both enc and dec)
  def forward(self,x):
    x = self.encoder(x) #encode the input data
    mu = self.mu_layer(x) #compute mean
    log_var = self.log_var_layer(x) #compute log variance
    z = self.reparameterize(mu,log_var)
    reconstructed_x = self.decoder(z) #decode it back to image
    return reconstructed_x, mu, log_var

In [ ]:
#define the loss fn and do the losss calculation
# VAE loss fn consist of two types of loss calculation
# 1. Reconstruction loss : checking if output is similar to input
# 2. KL Divergence loss : Ensure the learned latent space is following a normal distribution

def loss_fn(reconstructed_x, x, mu, log_var):
  #calculate the reconstruction loss
  recon_loss = nn.functional.binary_cross_entropy(reconstructed_x,x,reduction=sum)
  kl_loss = -0.5 * torch.sum(1+log_var-mu.pow(2)-log_var.exp())
  #return total loss
  return recon_loss + kl_loss